In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/tinyllama_qlora

!pwd

!ls


In [ ]:
# !pip uninstall -y -r requirements.txt

!pip install -r requirements.txt

In [ ]:
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes

print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("PEFT         :", peft.__version__)
print("TRL          :", trl.__version__)
print("Accelerate   :", accelerate.__version__)
print("BitsAndBytes :", bitsandbytes.__version__)

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# *LOAD AND FILTER DATA SET*

In [ ]:
from src.data.loader import load_data

DATASET_NAME = "databricks/databricks-dolly-15k"

dataset = load_data(DATASET_NAME)

print(dataset[0])

In [ ]:
from src.data.to_chat_format import format_chat

new_data_set = dataset.map(format_chat)

print(new_data_set[0])

In [ ]:
from src.config.tokenizer_loader import load_tokenizer
from src.data.apply_template import chat_template

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = load_tokenizer(MODEL_NAME)

templated_data = new_data_set.map(
    lambda x: chat_template(x, tokenizer)
)

print(templated_data[0])

In [ ]:
from src.data.filter_data import remove

training_dataset = remove(templated_data)

print(training_dataset[0])

# *LOAD MODEL USING LORA*

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
from src.config.bits_bytes_config import bit_byte_config

bnb_config = bit_byte_config()

print(bnb_config)

In [ ]:
from src.model.load_model import load_model

model = load_model(MODEL_NAME, bnb_config)

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
from src.config.lora_config import config_lora

lora_config = config_lora()

In [ ]:
from src.model.attach_lora import attach_lora

model = attach_lora(model, lora_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

# *TRAINING OF MODEL*

In [ ]:
from src.config.training_argument import training_argument

training_arg = training_argument()

In [ ]:
from src.training.load_trainer import load_trainer

trainer = load_trainer(model, training_dataset, training_arg)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("outputs/final_adapter")
tokenizer.save_pretrained("outputs/final_adapter")

In [ ]:
import json

src = "../outputs/final_adapter/trainer_state.json"

with open(src, "r") as f:
    trainer_state=json.load(f)

logs = trainer_state["log_history"]

In [ ]:
steps = []
losses = []

for log in logs:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

In [ ]:
!pip install matplotlib

In [ ]:


import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(steps, losses, marker="o")
plt.title("Training Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.grid(True)
plt.show()